In [ ]:
import os
import re
import math
import time
import nltk
import sympy
import shutil
import string
import random
import pickle
import pandas as pd
import requests
import networkx as nx
import pytesseract
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from PIL import Image
from bs4 import BeautifulSoup
from nltk.tag import pos_tag
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from selenium import webdriver
from nltk.stem import WordNetLemmatizer
from bing_image_downloader.downloader import download
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from nltk.corpus import wordnet as wn
from calculation import *
from word2image import *

resize_images_dir = "./Images/resize_images"

# Functions

In [ ]:
# Function to convert POS tags to WordNet format
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return None

# Function to process text: tokenize, POS tag, lemmatize
def process_text(text):
    lemmatizer = WordNetLemmatizer()

    # Tokenize the text
    tokens = word_tokenize(text)

    # POS tagging
    tagged_tokens = pos_tag(tokens)

    processed_tokens = []
    lemmatized_to_original = {}

    for word, tag in tagged_tokens:
        wordnet_pos = get_wordnet_pos(tag)
        if wordnet_pos:
            # Lemmatize the word
            lemmatized_word = lemmatizer.lemmatize(word, pos=wordnet_pos)
            # Add only significant POS
            processed_tokens.append(lemmatized_word)
            lemmatized_to_original[lemmatized_word] = word

    return processed_tokens, lemmatized_to_original

In [ ]:
def text2colorRatioDict(text):
    # Process the input text to get lemmatized words and a mapping to original words
    processed_word_list,lemmatized_to_original= process_text(text)
    text_color_ratio_dict = {}
    for lemmatized_word in processed_word_list:
        if word2image(lemmatized_word,10000,original_dir="./Images/original_images", temp_dir="./Images/temp", resize_dir="./Images/resize_images"):
            hue = extracthue(resize_images_dir, lemmatized_word)
            lightness = extractlightness(resize_images_dir, lemmatized_word)
            _, chroma_color_ratio, achroma_color_ratio = color_ratio_for_each_term(hue, lightness)
            color_ratio_dict = {**chroma_color_ratio, **achroma_color_ratio}
            text_color_ratio_dict[lemmatized_word] = color_ratio_dict
        else:
            continue
    return text_color_ratio_dict, lemmatized_to_original

In [ ]:
# Function to visualize text colors based on color ratios
def visualize_text_colors(text):
    # Get color ratios and original words from the input text
    color_ratio_dict, lemmatized_to_original = text2colorRatioDict(text)
    
    # Define chroma and achroma colors
    chroma_colors = ['red', 'orange', 'yellow', 'green', 'blue', 'purple', 'pink']
    achroma_colors = ['black', 'grey', 'white']
    
    # Split text into sentences based on punctuation (e.g., period)
    sentences = [sentence.strip() for sentence in text.split('.') if sentence]
    
    # Set up the plot
    fig, ax = plt.subplots(figsize=(12, 2 * len(sentences)))
    fig.patch.set_facecolor('#B3E5FC')  # Set background color for the entire plot
    ax.set_facecolor('#B3E5FC')  # Set background color for the axis
    ax.axis('off')  # Hide the axis
    
    # Iterate through each sentence and word
    for j, sentence in enumerate(sentences):
        word_list = sentence.split()
        processed_word_list, lemmatized_to_original = process_text(sentence)
        original_to_lemmatized = {v.lower(): k for k, v in lemmatized_to_original.items()}
        
        for i, word in enumerate(word_list):
            lower_word = word.lower().strip('.,')
            if lower_word in original_to_lemmatized:
                lemmatized_word = original_to_lemmatized[lower_word]
                if lemmatized_word in color_ratio_dict:
                    # Get color ratios for chroma and achroma colors
                    chroma_ratios = [color_ratio_dict[lemmatized_word].get(color, 0) for color in chroma_colors]
                    achroma_ratios = [color_ratio_dict[lemmatized_word].get(color, 0) for color in achroma_colors]
                    
                    left = i * 1.5
                    # Plot chroma color ratios
                    for ratio, color in zip(chroma_ratios, chroma_colors):
                        ax.barh(-(j * 2 + 0.1), ratio, color=color, left=left, height=0.4)
                        left += ratio
                        
                    left = i * 1.5
                    # Plot achroma color ratios
                    for ratio, color in zip(achroma_ratios, achroma_colors):
                        ax.barh(-(j * 2 + 0.6), ratio, color=color, left=left, height=0.4)
                        left += ratio
                    
            # Add text to the plot
            ax.text(i * 1.5 + 0.5, -(j * 2 - 0.3), word, ha='center', va='center', fontsize=12, color='black')
    
    # Display the plot
    plt.show()


# Applications

In [ ]:
text="The apparition of these faces in the crowd. Petals on a wet, black bough."

In [ ]:
text2colorRatioDict(text)[0]

In [ ]:
visualize_text_colors(text)